In [1]:
import os
import requests

from dotenv import load_dotenv
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass('Voer je OpenAI API key in: ')

In [2]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

In [3]:
llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Get current weather information for a city.
    """

    # Geocoding: stad -> latitude/longitude
    geo_url = (
        f"https://geocoding-api.open-meteo.com/v1/search"
        f"?name={city}&count=1"
    )

    geo_response = requests.get(geo_url)
    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"Could not find city: {city}"

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    city_name = location["name"]
    country = location["country"]

    # Weather ophalen
    weather_url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        f"&current_weather=true"
    )

    weather_response = requests.get(weather_url)
    weather_data = weather_response.json()

    current_weather = weather_data["current_weather"]

    return (
        f"Weather in {city_name}, {country}:\n"
        f"Temperature: {current_weather['temperature']}°C\n"
        f"Windspeed: {current_weather['windspeed']} km/h\n"
        f"Weather code: {current_weather['weathercode']}"
    )

tools = [get_weather]

In [5]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful assistant.

    Use the weather tool whenever weather information is needed.
    """,
)

In [6]:
task = """
What is the current weather in Amsterdam?
"""

# task = "What is the weather in Tokyo?"
# task = "Is it currently windy in New York?"
# task = "Compare the weather in Paris and Rome."

In [7]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": task}
    ]
})

print(response["messages"][-1].content)

The current weather in Amsterdam is as follows:
- Temperature: 13.2°C
- Windspeed: 22.0 km/h
- Weather condition: Cloudy (Weather code: 2)
